In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
import torch, os
from config import Config, AblationVariant
from model import Generator
from metrics import evaluate_full_test_set
import torch.serialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = Config()
torch.serialization.add_safe_globals([Config])

Mounted at /content/drive


In [ ]:
def load_ablation_generator(variant_name, device, cfg):
    ckpt_dir = os.path.join(cfg.ablation_dir, variant_name, 'checkpoints')
    ckpt_path = os.path.join(ckpt_dir, 'latest.pth')
    assert os.path.exists(ckpt_path), f"No checkpoint for '{variant_name}' yet — run 02_Ablation_Training.ipynb with VARIANT='{variant_name}' first."
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f"{variant_name}: loaded epoch {ck['epoch']} (flags={ck['flags']})")
    G = Generator(cfg.img_size, num_domains=cfg.num_domains).to(device)
    G.load_state_dict(ck['G_state_dict'])
    G.eval()
    return G, ck['epoch']

variants_to_eval = ['no_cls', 'no_cycle', 'no_identity', 'no_gp']
ablation_summaries = []

for variant in variants_to_eval:
    try:
        G, epoch = load_ablation_generator(variant, device, cfg)
    except AssertionError as e:
        print("SKIPPING:", e)
        continue
    per_item, summary, fid_rows = evaluate_full_test_set(
        G, cfg, device, out_dir=cfg.ablation_dir, model_name=f'ablation_{variant}'
    )
    summary['variant'] = variant
    summary['trained_epochs'] = epoch
    ablation_summaries.append(summary)
    del G
    torch.cuda.empty_cache()

no_cls: loaded epoch 100 (flags={'use_cls': False, 'use_cycle': True, 'use_identity': True, 'use_gp': True, 'use_perceptual': True})
[test] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 190MB/s] 


[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[ablation_no_cls] n_translations=1000 (should be 250x4= 1000)
[ablation_no_cls] PSNR 44.49±3.08 | SSIM 0.992±0.004 | MSE 0.00008±0.00061 | FID 19.50±5.80
no_cycle: loaded epoch 100 (flags={'use_cls': True, 'use_cycle': False, 'use_identity': True, 'use_gp': True, 'use_perceptual': True})
[test] domains found: {'Diabetic Retinopa

## Include the full model (already trained — no retraining needed)
This reuses whatever `EyeGAN_per_item_results.csv` / summary Notebook 01 produced, so the "full model" row in the ablation table is the exact same run as everywhere else in the revision, not a re-training.

In [ ]:
import pandas as pd

full_csv = os.path.join(cfg.eval_dir, 'EyeGAN_per_item_results.csv')
assert os.path.exists(full_csv), "Run 01_FullTestSet_Evaluation.ipynb first so the full-model numbers exist."
full_df = pd.read_csv(full_csv)
fid_full = pd.read_csv(os.path.join(cfg.eval_dir, 'EyeGAN_fid_per_domain.csv'))['fid'].mean()

full_row = {
    'model': 'EyeGAN', 'variant': 'full_model', 'trained_epochs': 100,
    'n_translations': len(full_df),
    'psnr_mean': full_df.psnr.mean(), 'psnr_std': full_df.psnr.std(),
    'ssim_mean': full_df.ssim.mean(), 'ssim_std': full_df.ssim.std(),
    'mse_mean': full_df.mse.mean(), 'mse_std': full_df.mse.std(),
    'fid_mean': fid_full, 'fid_std': float('nan'),
}
ablation_summaries.append(full_row)

In [ ]:
ablation_table = pd.DataFrame(ablation_summaries)
ablation_table = ablation_table[['variant', 'trained_epochs', 'psnr_mean', 'psnr_std',
                                  'ssim_mean', 'ssim_std', 'mse_mean', 'mse_std', 'fid_mean']]
ablation_table.columns = ['Variant (loss removed)', 'Epochs', 'PSNR mean', 'PSNR std',
                           'SSIM mean', 'SSIM std', 'MSE mean', 'MSE std', 'FID']
name_map = {
    'full_model': 'Full EyeGAN (nothing removed)',
    'no_cls': 'w/o domain classification loss',
    'no_cycle': 'w/o cycle-consistency loss',
    'no_identity': 'w/o identity loss',
    'no_gp': 'w/o gradient penalty',
}
ablation_table['Variant (loss removed)'] = ablation_table['Variant (loss removed)'].map(name_map)
ablation_table = ablation_table.sort_values('FID')
print(ablation_table.round(3).to_string(index=False))

out_path = os.path.join(cfg.ablation_dir, 'ablation_table.csv')
ablation_table.to_csv(out_path, index=False)
print(f"\nSaved to {out_path} — this is your new Table for the ablation study "
      "(R1-Q5, R2-Q3c). The row with the worst FID/PSNR/SSIM tells you which "
      "loss term contributes most; discuss that in the response letter.")

        Variant (loss removed)  Epochs  PSNR mean  PSNR std  SSIM mean  SSIM std  MSE mean  MSE std    FID
          w/o gradient penalty     100     39.535     1.979      0.979     0.006     0.000    0.000 19.198
w/o domain classification loss     100     44.488     3.079      0.992     0.004     0.000    0.001 19.496
 Full EyeGAN (nothing removed)     100     36.245     2.695      0.926     0.033     0.000    0.000 21.613
    w/o cycle-consistency loss     100     36.289     2.152      0.931     0.027     0.000    0.000 21.749
             w/o identity loss     100     32.514     1.932      0.864     0.042     0.001    0.000 23.782

Saved to /content/drive/MyDrive/CSE720/revision/ablation/ablation_table.csv — this is your new Table for the ablation study (R1-Q5, R2-Q3c). The row with the worst FID/PSNR/SSIM tells you which loss term contributes most; discuss that in the response letter.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_to_plot = [('PSNR mean', 'PSNR (dB, higher better)'),
                    ('SSIM mean', 'SSIM (higher better)'),
                    ('FID', 'FID (lower better)')]
for ax, (col, title) in zip(axes, metrics_to_plot):
    ax.bar(ablation_table['Variant (loss removed)'], ablation_table[col], color='steelblue')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
fig_path = os.path.join(cfg.ablation_dir, 'ablation_chart.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print("Saved figure to", fig_path)